# 🤖 MPT Agent System — Google Colab Runner
**Runtime: GPU (T4 free tier is enough)**

This notebook:
1. Clones MoneyPrinterTurbo + this agent system
2. Loads Gemma 3 4B locally via HuggingFace (no Ollama needed)
3. Runs the full pipeline: Trends → Script → Video (MPT) → SEO → Publish

## ⚙️ Step 1 — Install dependencies

In [ ]:
# MoneyPrinterTurbo dependencies
!git clone https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo 2>/dev/null || echo 'MPT already cloned'
!cd /content/MoneyPrinterTurbo && pip install -q -r requirements.txt

# Agent system
!git clone https://github.com/Bhushan-Khachane/mpt-agent-system.git /content/mpt-agent-system 2>/dev/null || (cd /content/mpt-agent-system && git pull)
!pip install -q feedparser pytrends requests google-api-python-client google-auth-httplib2 google-auth-oauthlib

# HuggingFace Transformers + bitsandbytes for 4-bit Gemma on T4
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
print('All dependencies installed.')

## 🔑 Step 2 — Configure keys & paths

In [ ]:
import os, sys

# ── Paths ────────────────────────────────────────────────────────────────────
os.environ['MPT_DIR']      = '/content/MoneyPrinterTurbo'
os.environ['MPT_USE_API']  = 'false'  # CLI mode (change to true if MPT server is running)
os.environ['MPT_API_URL']  = 'http://127.0.0.1:8080'

# ── LLM: force HuggingFace Gemma (auto-detected on Colab anyway) ─────────────
os.environ['LLM_BACKEND']  = 'hf_gemma'
os.environ['HF_MODEL_ID']  = 'google/gemma-3-4b-it'
os.environ['HF_TOKEN']     = ''  # only needed if model is gated; get free token at hf.co

# ── Pexels (free key at pexels.com/api) ──────────────────────────────────────
os.environ['PEXELS_API_KEY']  = 'YOUR_PEXELS_KEY'
os.environ['PIXABAY_API_KEY'] = 'YOUR_PIXABAY_KEY'

# ── YouTube / Instagram channel IDs (one set per niche) ──────────────────────
os.environ['YT_CHANNEL_AI']    = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_CLEAN'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_MONEY'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['IG_ACCOUNT_AI']    = 'your_ig_handle'
os.environ['IG_ACCOUNT_CLEAN'] = 'your_ig_handle'
os.environ['IG_ACCOUNT_MONEY'] = 'your_ig_handle'

# Add agent system to Python path
sys.path.insert(0, '/content/mpt-agent-system')

# Create required dirs
import pathlib
for d in ['data','videos','logs','credentials']:
    pathlib.Path(f'/content/mpt-agent-system/{d}').mkdir(exist_ok=True)

import json, pathlib
q = pathlib.Path('/content/mpt-agent-system/data/topics_queue.json')
s = pathlib.Path('/content/mpt-agent-system/data/seen_topics.json')
if not q.exists(): q.write_text('[]')
if not s.exists(): s.write_text('{}')

print('Config done.')

## 🧠 Step 3 — Pre-load Gemma 3 4B into GPU memory
This takes ~3-5 min on first run. Model is cached after that.

In [ ]:
import os
os.chdir('/content/mpt-agent-system')

from agents.llm_client import _load_hf_pipeline
pipe = _load_hf_pipeline()
print('Gemma loaded and ready!')

## 📡 Step 4 — TrendScout: fetch trending topics

In [ ]:
os.chdir('/content/mpt-agent-system')
from agents.trend_scout import run_trend_scout
topics = run_trend_scout()
print(f'\nTop 5 topics queued:')
for t in topics[:5]:
    print(f"  [{t['niche']}] {t['title']} (score={t['score_val']:.1f})")

## ✍️ Step 5 — ScriptWriter: generate scripts with Gemma

In [ ]:
from agents.script_writer import run_script_writer
run_script_writer(batch_size=6)

# Preview one script
import json
queue = json.loads(open('/content/mpt-agent-system/data/topics_queue.json').read())
scripted = [t for t in queue if t.get('status') == 'scripted']
if scripted:
    print(f"\n--- Sample script ({scripted[0]['niche']}) ---")
    print(scripted[0]['script'])

## 🎬 Step 6 — VideoProducer: generate videos via MoneyPrinterTurbo

In [ ]:
from agents.video_producer import run_video_producer
run_video_producer(batch_size=3, use_api=False)  # set use_api=True if MPT server is running

## 🔍 Step 7 — SEO Agent: generate titles, tags & captions

In [ ]:
from agents.seo_agent import run_seo_agent
run_seo_agent(batch_size=6)

## 🚀 Step 8 — Publisher: post to YouTube Shorts + Instagram + Facebook

In [ ]:
# Upload YouTube OAuth token for each channel before running.
# Place token files at: /content/mpt-agent-system/credentials/youtube_{niche}_token.json
from agents.publisher import run_publisher
run_publisher(batch_size=3)

## ⚡ Run full pipeline in one shot

In [ ]:
os.chdir('/content/mpt-agent-system')
!python orchestrator.py